In [3]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from constants import PATH_TO_FINAL_OUTPUT
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm


In [4]:
df = pd.read_csv(PATH_TO_FINAL_OUTPUT)

In [5]:
df['debate_date'] = pd.to_datetime(df['debate_date'])
df['speaker_first_debate_date'] = pd.to_datetime(df['speaker_first_debate_date'])

motion_balance = df[df['side'] == 'aff'].groupby(['debate_id', 'motion'])['ballots_gained'].mean().reset_index()
motion_balance = motion_balance.groupby('motion')['ballots_gained'].mean().reset_index()
motion_balance.columns = ['motion', 'motion_balance']

df = df.merge(motion_balance, on='motion', how='left')

df['years_since_first_debate'] = df['debate_date'].dt.year - df['speaker_first_debate_date'].dt.year

df['tournament_round'] = df.groupby('tournament_id')['debate_date'].rank(method='dense').astype(int)

df['is_aff'] = (df['side'] == 'aff').astype(int)
df['motion_balance_x_aff'] = df['motion_balance'] * df['is_aff']

In [9]:

df_reg = df.dropna(subset=['speaker_points', 'is_male', 'years_since_first_debate', 
                            'tournament_round', 'motion_balance_x_aff', 'motion_balance'])

X = df_reg[['is_male', 'years_since_first_debate', 'tournament_round', 'motion_balance_x_aff', 'motion_balance']].astype(float)
y = df_reg['speaker_points'].astype(float)

X = sm.add_constant(X)
model = sm.OLS(y, X).fit()

In [10]:
print("\n" + "="*80)
print("OLS Regression Results")
print("="*80)
print(model.summary())


OLS Regression Results
                            OLS Regression Results                            
Dep. Variable:         speaker_points   R-squared:                       0.203
Model:                            OLS   Adj. R-squared:                  0.197
Method:                 Least Squares   F-statistic:                     33.61
Date:                Mon, 19 Jan 2026   Prob (F-statistic):           1.28e-30
Time:                        22:43:13   Log-Likelihood:                -1914.6
No. Observations:                 667   AIC:                             3841.
Df Residuals:                     661   BIC:                             3868.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
